In [38]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from collections import Counter
import re
import matplotlib.pyplot as plt

# 固定随机种子
torch.manual_seed(42)
np.random.seed(42)

In [39]:
# 选 4 个类别
categories = ['rec.autos', 'sci.space', 'comp.graphics', 'talk.politics.misc']
print("Loading data...")
newsgroups = fetch_20newsgroups(subset='all', categories=categories, shuffle=True, random_state=42)

texts = newsgroups.data
labels = newsgroups.target

# 每类最多取 500 个
max_per_class = 500
balanced_texts, balanced_labels = [], []
for c in range(len(categories)):
    idx = [i for i, label in enumerate(labels) if label == c][:max_per_class]
    balanced_texts.extend([texts[i] for i in idx])
    balanced_labels.extend([labels[i] for i in idx])

texts = balanced_texts
labels = np.array(balanced_labels)

print(f"Total samples: {len(texts)}")

Loading data...
Total samples: 2000


In [40]:
X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels, test_size=0.2, stratify=labels, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
print("Train class distribution:", np.bincount(y_train))
print("Val class distribution:", np.bincount(y_val))
print("Test class distribution:", np.bincount(y_test))

Train: 1600, Val: 200, Test: 200
Train class distribution: [400 400 400 400]
Val class distribution: [50 50 50 50]
Test class distribution: [50 50 50 50]


In [41]:
def simple_tokenizer(text):
    # 小写 + 非字母数字替换成空格
    text = text.lower()
    text = re.sub(r'[^a-z0-9]', ' ', text)
    # 合并多个空格
    text = re.sub(r'\s+', ' ', text).strip()
    return text.split()

# 训练集分词
train_tokens = [simple_tokenizer(t) for t in X_train]

# 构建词频表
word_freq = Counter()
for tokens in train_tokens:
    word_freq.update(tokens)

# top-K = 5000
K = 5000
most_common = word_freq.most_common(K)
vocab = {word: idx+1 for idx, (word, _) in enumerate(most_common)}  # idx 0 留给 <unk>
vocab['<unk>'] = 0
vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")

def encode_text(tokens, vocab):
    return [vocab.get(t, 0) for t in tokens]

# 示例
example_tokens = train_tokens[0]
example_ids = encode_text(example_tokens, vocab)
print("Example tokens:", example_tokens[:20])
print("Example ids:", example_ids[:20])

Vocab size: 5001
Example tokens: ['from', 'rjwade', 'rainbow', 'ecn', 'purdue', 'edu', 'robert', 'j', 'wade', 'subject', 're', 'saturn', 'extended', 'warranty', 'organization', 'purdue', 'university', 'engineering', 'computer', 'network']
Example ids: [15, 0, 0, 1050, 852, 16, 776, 330, 4093, 34, 32, 997, 2128, 3620, 36, 852, 76, 483, 198, 777]


In [42]:
# 计算文档频率 DF (训练集)
df = np.zeros(vocab_size)
for tokens in train_tokens:
    unique_ids = set(encode_text(tokens, vocab))
    for uid in unique_ids:
        df[uid] += 1

# 平滑 IDF: log((N+1)/(df+1)) + 1
N_train = len(train_tokens)
idf = np.log((N_train + 1) / (df + 1)) + 1

def bow_tfidf(tokens, vocab, idf):
    ids = encode_text(tokens, vocab)
    # TF: 词频 / 长度（归一化）
    tf = np.zeros(vocab_size)
    for idx in ids:
        tf[idx] += 1
    tf = tf / (len(ids) + 1e-8)
    # TF-IDF
    tfidf = tf * idf
    # L2 归一化
    norm = np.linalg.norm(tfidf)
    if norm > 0:
        tfidf = tfidf / norm
    return tfidf

# 测试一个样本
bow_example = bow_tfidf(train_tokens[0], vocab, idf)
print("BoW feature dimension:", len(bow_example))

BoW feature dimension: 5001


In [43]:
class BowSoftmaxClassifier:
    def __init__(self, input_dim, num_classes, lr=1e-3):
        self.W = np.random.randn(input_dim, num_classes) * 0.01
        self.b = np.zeros(num_classes)
        self.lr = lr

    def softmax(self, logits):
        exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
        return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)

    def forward(self, X):
        return X @ self.W + self.b

    def compute_loss(self, X, y):
        logits = self.forward(X)
        probs = self.softmax(logits)
        n = X.shape[0]
        # cross-entropy loss
        correct_log_probs = -np.log(probs[range(n), y])
        loss = np.mean(correct_log_probs)
        return loss, probs

    def backward(self, X, y, probs):
        n = X.shape[0]
        dlogits = probs.copy()
        dlogits[range(n), y] -= 1
        dlogits /= n
        dW = X.T @ dlogits
        db = np.sum(dlogits, axis=0)
        return dW, db

    def update(self, dW, db):
        self.W -= self.lr * dW
        self.b -= self.lr * db

    def predict(self, X):
        logits = self.forward(X)
        return np.argmax(logits, axis=1)

# 准备训练数据
X_bow_train = np.array([bow_tfidf(t, vocab, idf) for t in train_tokens])
X_bow_val = np.array([bow_tfidf(simple_tokenizer(t), vocab, idf) for t in X_val])
X_bow_test = np.array([bow_tfidf(simple_tokenizer(t), vocab, idf) for t in X_test])

print("Train BoW shape:", X_bow_train.shape)

Train BoW shape: (1600, 5001)


In [44]:
# 训练
model_bow = BowSoftmaxClassifier(vocab_size, len(categories), lr=1e-3)

epochs = 10
for epoch in range(epochs):
    loss, probs = model_bow.compute_loss(X_bow_train, y_train)
    dW, db = model_bow.backward(X_bow_train, y_train, probs)
    model_bow.update(dW, db)

    # 验证
    y_val_pred = model_bow.predict(X_bow_val)
    val_acc = np.mean(y_val_pred == y_val)
    print(f"Epoch {epoch+1}, loss: {loss:.4f}, val_acc: {val_acc:.4f}")

# 测试
y_test_pred = model_bow.predict(X_bow_test)
test_acc = np.mean(y_test_pred == y_test)
print(f"Test accuracy: {test_acc:.4f}")

Epoch 1, loss: 1.3866, val_acc: 0.2150
Epoch 2, loss: 1.3866, val_acc: 0.2150
Epoch 3, loss: 1.3866, val_acc: 0.2150
Epoch 4, loss: 1.3866, val_acc: 0.2150
Epoch 5, loss: 1.3866, val_acc: 0.2150
Epoch 6, loss: 1.3866, val_acc: 0.2150
Epoch 7, loss: 1.3866, val_acc: 0.2150
Epoch 8, loss: 1.3866, val_acc: 0.2150
Epoch 9, loss: 1.3866, val_acc: 0.2150
Epoch 10, loss: 1.3866, val_acc: 0.2150
Test accuracy: 0.2750


In [45]:
# 每个类别 top-10 词
inv_vocab = {v: k for k, v in vocab.items()}
for c in range(len(categories)):
    top_indices = np.argsort(model_bow.W[:, c])[-10:][::-1]
    top_words = [inv_vocab[i] for i in top_indices if i in inv_vocab]
    print(f"\nClass {categories[c]} top-10 words:")
    print(top_words)


Class rec.autos top-10 words:
['causes', 'mounted', 'antenna', '41', 'referred', 'transport', 'los', 'moves', 'writer', 'gear']

Class sci.space top-10 words:
['out', '9760', 'sort', 'play', 'specific', 'existence', 'fluid', 'write', 'beowulf', 'inevitable']

Class comp.graphics top-10 words:
['sub', 'replies', 'telling', 'canadian', 'careful', 'want', 'learn', 'asked', 'hopkins', 'uncle']

Class talk.politics.misc top-10 words:
['tennessee', 'rocks', 'king', 'bet', 'speed', 'stimulus', 'participating', 'signals', 'jump', 'tower']


In [46]:
class EmbeddingAvgModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, ids, lengths):
        # ids: (batch, seq_len)
        emb = self.embedding(ids)  # (batch, seq_len, embed_dim)
        # mask 平均
        mask = (ids != 0).float().unsqueeze(-1)  # (batch, seq_len, 1)
        sum_emb = (emb * mask).sum(dim=1)        # (batch, embed_dim)
        avg_emb = sum_emb / lengths.unsqueeze(1).float()
        return self.fc(avg_emb)

# 准备 padding 数据
def collate_fn(batch):
    texts_batch, labels_batch = zip(*batch)
    ids_batch = [encode_text(simple_tokenizer(t), vocab) for t in texts_batch]
    lengths = torch.tensor([len(ids) for ids in ids_batch])
    max_len = max(lengths)
    padded_ids = torch.zeros(len(batch), max_len, dtype=torch.long)
    for i, ids in enumerate(ids_batch):
        padded_ids[i, :len(ids)] = torch.tensor(ids)
    return padded_ids, lengths, torch.tensor(labels_batch)

train_dataset = list(zip(X_train, y_train))
val_dataset = list(zip(X_val, y_val))
test_dataset = list(zip(X_test, y_test))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, collate_fn=collate_fn)

model_emb = EmbeddingAvgModel(vocab_size, embed_dim=64, num_classes=len(categories))
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_emb.parameters(), lr=1e-3)

# 训练
for epoch in range(10):
    model_emb.train()
    total_loss = 0
    for ids, lengths, labels in train_loader:
        optimizer.zero_grad()
        logits = model_emb(ids, lengths)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # 验证
    model_emb.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for ids, lengths, labels in val_loader:
            logits = model_emb(ids, lengths)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += len(labels)
    val_acc = correct / total
    print(f"Epoch {epoch+1}, loss: {total_loss/len(train_loader):.4f}, val_acc: {val_acc:.4f}")

# 测试
model_emb.eval()
correct = 0
total = 0
with torch.no_grad():
    for ids, lengths, labels in test_loader:
        logits = model_emb(ids, lengths)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += len(labels)
print(f"Test accuracy: {correct/total:.4f}")

Epoch 1, loss: 1.3825, val_acc: 0.2450
Epoch 2, loss: 1.3468, val_acc: 0.4200
Epoch 3, loss: 1.3033, val_acc: 0.5550
Epoch 4, loss: 1.2455, val_acc: 0.6750
Epoch 5, loss: 1.1697, val_acc: 0.7250
Epoch 6, loss: 1.0784, val_acc: 0.7450
Epoch 7, loss: 0.9767, val_acc: 0.7800
Epoch 8, loss: 0.8718, val_acc: 0.8200
Epoch 9, loss: 0.7697, val_acc: 0.8250
Epoch 10, loss: 0.6747, val_acc: 0.8600
Test accuracy: 0.8100
